In [2]:
import polars as pl
from pathlib import Path
import pandas as pd


In [22]:
maps= {
    'model-odi7xtno:v2': '/home/nikita/carGPT/outputs/2025-01-20/13-22-19/predictions', # no_wpts | full ds
    'model-n8693bg7:v2': '/home/nikita/carGPT/outputs/2025-01-20/13-27-47/predictions', # wpts | full ds
    }

model_name = 'model-n8693bg7:v2'
d = Path(maps[model_name])
f_list = sorted(d.glob('*.parquet'), key=lambda x: int(x.stem))
print(len(f_list))

7710


In [23]:
from tqdm import tqdm
df_res = pd.DataFrame()

for f in tqdm(f_list):
    df = pl.read_parquet(f)
    for name in ('gas_pedal', 'brake_pedal', 'steering_angle'):
        df_res = pd.concat([df_res, pd.DataFrame([{
            'frame_idx': df['batch/data/ImageMetadata.cam_front_left.frame_idx'][0][-1],
            'timestamp': df['batch/data/ImageMetadata.cam_front_left.time_stamp'][0][-1],
            'drive_id': df['batch/meta/input_id'][0],
            'name': name,
            'ground_truth': df[f'inputs/continuous/{name}'][0][-1][0],
            'prediction': df[f'predictions/policy/prediction/continuous/{name}'][0][-1][0],
            'prediction_std': df[f'predictions/policy/prediction_std/continuous/{name}'][0][-1][0],
            'prediction_probs': df[f'predictions/policy/prediction_probs/continuous/{name}'][0][-1][0],
            'score_logprob': df[f'predictions/policy/score_logprob/continuous/{name}'][0][-1],
            'score_l1': df[f'predictions/policy/score_l1/continuous/{name}'][0][-1][0],
            'datetime': pd.to_datetime(df['batch/data/ImageMetadata.cam_front_left.time_stamp'][0][-1], unit='ns')
        }])], ignore_index=True)

path_to_save = Path(f'/nas/team-space/nikita/artifacts/{model_name}/scores.csv')
path_to_save.parent.mkdir(parents=True, exist_ok=True)
df_res.to_csv(path_to_save, index=False)


100%|██████████| 7710/7710 [00:18<00:00, 407.39it/s]


# Viz

In [3]:
# df_wpts = pd.read_csv('/nas/team-space/nikita/artifacts/model-fxi8l52t:v22/scores.csv')
# df_no_wpts = pd.read_csv('/nas/team-space/nikita/artifacts/model-uzrdf384:v22/scores.csv')
df_wpts = pd.read_csv('/nas/team-space/nikita/artifacts/model-n8693bg7:v2/scores.csv')
df_no_wpts = pd.read_csv('/nas/team-space/nikita/artifacts/model-odi7xtno:v2/scores.csv')


In [5]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# drive_id = 'Niro096-HQ/2023-01-11--13-47-36' # train
# drive_id = 'Niro115-HQ/2023-05-16--10-47-33' # val
drive_id = "G1-00428/2024-04-23--07-00-31" # test
# action = 'gas_pedal'
action = 'steering_angle'

fig = make_subplots(rows=2, 
                    cols=1, 
                    shared_xaxes=True, 
                    vertical_spacing=0.1, 
                    subplot_titles=('Steering Angle Prediction Std', 'Score L1'),
                    )
df_wpts_steering = df_wpts[(df_wpts['name'] == action) & (df_wpts['drive_id'] == drive_id)]
df_no_wpts_steering = df_no_wpts[(df_no_wpts['name'] == action) & (df_no_wpts['drive_id'] == drive_id)]

# Top subplot
fig.add_trace(go.Scatter(x=df_wpts_steering['frame_idx'], y=df_wpts_steering['prediction_std'], mode='lines', name='wpts', line=dict(color='red')), row=1, col=1)
fig.add_trace(go.Scatter(x=df_no_wpts_steering['frame_idx'], y=df_no_wpts_steering['prediction_std'], mode='lines', name='no_wpts', line=dict(color='blue')), row=1, col=1)

# Bottom subplot
fig.add_trace(go.Scatter(x=df_wpts_steering['frame_idx'], y=df_wpts_steering['score_l1'], mode='lines', name='wpts', line=dict(color='red')), row=2, col=1)
fig.add_trace(go.Scatter(x=df_no_wpts_steering['frame_idx'], y=df_no_wpts_steering['score_l1'], mode='lines', name='no_wpts', line=dict(color='blue')), row=2, col=1)

fig.update_layout(title=f'{action} Prediction Analysis', 
                  xaxis_title='Frame Index', 
                  yaxis_title='Prediction Std',
                  height=600,  # Increase the height
                  font=dict(size=18)  # Increase the font size
                 )
fig.update_yaxes(title_text='Prediction Std', row=1, col=1)
fig.update_yaxes(title_text='Score L1', row=2, col=1)

fig.show()
from prettytable import PrettyTable

# Create a PrettyTable object
table = PrettyTable()

# Add columns
table.field_names = ["Metric", "wpts", "no_wpts"]

# Add rows
table.add_row(["Score L1 Mean", f"{df_wpts_steering['score_l1'].mean():.4f}", f"{df_no_wpts_steering['score_l1'].mean():.4f}"])
table.add_row(["Prediction Std Mean", f"{df_wpts_steering['prediction_std'].mean():.4f}", f"{df_no_wpts_steering['prediction_std'].mean():.4f}"])

# Print the table
print(table)


+---------------------+--------+---------+
|        Metric       |  wpts  | no_wpts |
+---------------------+--------+---------+
|    Score L1 Mean    | 0.0385 |  0.0458 |
| Prediction Std Mean | 0.0406 |  0.0485 |
+---------------------+--------+---------+
